# File System Ops

## Quick notes

### What this covers
- Testing files (`[ -f ]`, `-d`, `-e`, `-L`, `-nt` ...), listing (`ls -l`), creating trees (`mkdir -p`, brace expansion).
- Copy, move, delete semantics and their BSD/macOS flags, and safe-delete habits.
- Globbing (`* ? [..]`, dotfiles, `extglob`, `nullglob`), `find` in depth, `chmod`, `umask`, `stat`.
- Symlinks vs hard links, `touch` with fixed times, `du`/`df`, `basename`/`dirname`, `mktemp` + `trap`, `file`.

### Key points
- Everything here happens inside a private sandbox folder; nothing outside it is touched.
- `[ -e f ]` exists, `-f` regular file, `-d` directory, `-L` symlink, `-r -w -x` permissions, `-s` non-empty, `-nt`/`-ot` newer/older.
- `mkdir -p a/b/c` makes parents and does not fail if present. Brace expansion `{a,b}` builds many paths at once.
- `find` can filter by name, type, age, size, permission and run commands. `-print0 | xargs -0` and `-exec ... +` are the safe ways to pass names on.
- Permissions are 3 octal digits (owner/group/other). `umask` removes bits from new files.
- A symlink is a pointer by name; a hard link is a second name for the same data.

### Gotchas (the part worth revising twice)
- **BSD `stat`** uses `-f` formats (`stat -f '%Lp %N' f`); GNU uses `-c` (`stat -c '%a %n' f`). The same script will not run on both.
- **`sed -i`** and **`date -d`** style GNU habits fail on macOS; BSD `touch -d` only takes `YYYY-MM-DDThh:mm:SS`, so GNU forms like `'2020-01-01'` or `'2 days ago'` fail; `touch -t 202001010000` works on both.
- `cp -r dir newdir` behaves differently if `newdir` already exists (copies INTO it). A trailing slash on the source can behave differently on BSD and GNU (on BSD `cp -R src/ dst` copies the contents of `src`), so avoid relying on it.
- `rm -rf "$dir"/*` with an empty `$dir` becomes `rm -rf /*`. Guard with `"${dir:?}"`.
- `find ... -delete` placed BEFORE the tests deletes everything it walks. Put `-delete` last.
- `{01..03}` in bash 3.2 does NOT zero-pad (bash 4+ gives 01 02 03). Use `printf '%02d'` or a loop for padding.
- `readlink -f` and `realpath` exist on this macOS, but not on older macOS releases; portable: `(cd "$(dirname "$p")" && pwd -P)`.
- `ls` output is for humans: do not parse it. Use globs or `find`.
- `mv -n` / `cp -n` (no clobber) silently skip existing targets; check the result.

### When it matters
- Install/backup/cleanup scripts, build pipelines, anything that deletes or moves data.

### One-line recap example
```bash
find "$dir" -type f -name '*.tmp' -mtime +7 -print0 | xargs -0 rm -f --
```

## 01. The private sandbox

- The notebook runs in a fresh, empty folder. We make one project tree here and every later section works inside it.
- `mkdir -p` with brace expansion builds the whole tree in one command.
- Brace expansion happens BEFORE variables and globs: it just writes out text.
- Always check with `find ... | sort` (order from `find` is not fixed).

In [1]:
# setup: a %%bash cell magic that keeps ONE bash session alive between cells
import atexit, os, shutil, subprocess, tempfile, uuid
from IPython.core.magic import register_cell_magic

_WORK = tempfile.mkdtemp(prefix="bashnb_")
atexit.register(shutil.rmtree, _WORK, ignore_errors=True)
_ENV = {k: v for k, v in os.environ.items() if k not in ("CLICOLOR", "CLICOLOR_FORCE", "FORCE_COLOR", "PYDEVD_DISABLE_FILE_VALIDATION")}
_ENV.update(LC_ALL="C", TZ="UTC", HOME=_WORK, PS1="", PS4="+ ")
_session = None

def _start():
    global _session
    _session = subprocess.Popen(["bash", "--noprofile", "--norc"], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=_WORK, env=_ENV)

def _clean(text):
    return text.replace(os.path.join(_WORK, "script"), "script").replace(_WORK, "<work>")

def _run_persistent(code, timeout=60):
    global _session
    if _session is None or _session.poll() is not None:
        _start()
    token = uuid.uuid4().hex
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    _session.stdin.write(". '%s' </dev/null; printf '\\n%s %%d\\n' $?\n" % (path, token))
    _session.stdin.flush()
    out, status = [], None
    import threading
    result = {}
    def reader():
        while True:
            line = _session.stdout.readline()
            if not line:
                result["status"] = None; return
            if line.startswith(token + " "):
                result["status"] = int(line.split()[1]); return
            out.append(line)
    t = threading.Thread(target=reader, daemon=True); t.start(); t.join(timeout)
    os.remove(path)
    if t.is_alive():
        _session.kill(); _session = None
        return "".join(out) + "[timed out]\n", 124
    if result.get("status") is None:       # the shell itself exited (e.g. the cell ran `exit N`)
        code = _session.wait(); _session = None
        return "".join(out) + "(the bash session ended; the next cell starts a fresh one)\n", code
    text = "".join(out)
    if text.endswith("\n\n"):
        text = text[:-1]          # drop the blank line our status marker adds
    return text, result.get("status") if result.get("status") is not None else 1

def _run_fresh(code, timeout=60):
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    done = subprocess.run(["bash", "--noprofile", "--norc", path], stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, cwd=_WORK, env=_ENV, timeout=timeout)
    os.remove(path)
    return done.stdout, done.returncode

@register_cell_magic
def bash(line, cell):
    """Run the cell in bash. Flags: fresh (new shell), expect_fail (show non-zero status instead of raising)."""
    flags = line.split()
    text, status = (_run_fresh if "fresh" in flags else _run_persistent)(cell)
    print(_clean(text), end="" if text.endswith("\n") or not text else "\n")
    if status != 0:
        if "expect_fail" in flags:
            print("[exit status %d]" % status)
        else:
            raise RuntimeError("bash cell exited with status %d" % status)

In [2]:
%%bash
unset CLICOLOR_FORCE CLICOLOR        # Jupyter sets colours for ls; plain output reads better
mkdir fs && cd fs
ROOT=$PWD                          # every later cell starts with: cd "$ROOT"
mkdir -p proj/{src,docs,tests}/{a,b}
mkdir -p proj/empty_dir
echo 'print("hi")' > proj/src/a/main.py
echo 'def f(): pass' > proj/src/b/util.py
echo '# Readme' > proj/docs/README.md
echo 'notes' > proj/docs/a/notes.txt
: > proj/tests/a/test_main.py
find proj | sort

proj
proj/docs
proj/docs/README.md
proj/docs/a
proj/docs/a/notes.txt
proj/docs/b
proj/empty_dir
proj/src
proj/src/a
proj/src/a/main.py
proj/src/b
proj/src/b/util.py
proj/tests
proj/tests/a
proj/tests/a/test_main.py
proj/tests/b


## 02. Brace expansion details

- `{a,b,c}` lists items; `{1..5}` counts; `{a..e}` letters; combine with text: `file{1,2}.txt`.
- Braces can nest and multiply: `{x,y}{1,2}` gives x1 x2 y1 y2.
- bash 3.2 does not zero-pad `{01..03}`; bash 4 would give `01 02 03`. `{1..10..2}` (step) is also bash 4+.
- Braces do not check files exist (unlike globs).

In [3]:
%%bash
cd "$ROOT"
echo file{1,2,3}.txt
echo {a..e}
echo {x,y}{1,2}
echo "range with leading zero on 3.2:" {01..03}
echo "portable padding:" $(printf '%02d ' 1 2 3)
echo "no files needed for braces:" /nonexistent/{a,b}

file1.txt file2.txt file3.txt
a b c d e
x1 x2 y1 y2
range with leading zero on 3.2: 1 2 3
portable padding: 01 02 03
no files needed for braces: /nonexistent/a /nonexistent/b


In [4]:
%%bash
cd "$ROOT"
if (( BASH_VERSINFO[0] >= 4 )); then eval 'echo {1..10..3}'; else echo "skipped: step in brace ranges needs bash 4+"; fi

skipped: step in brace ranges needs bash 4+


## 03. Test operators: `[ ]` and `[[ ]]`

- File tests: `-e` exists, `-f` regular file, `-d` directory, `-L` symlink, `-s` size > 0, `-r -w -x` access for you.
- Comparisons: `a -nt b` (a newer than b), `a -ot b` (older), `a -ef b` (same file).
- `[[ ... ]]` is a bash keyword (not POSIX): no word splitting inside it, and it also accepts `&&`, `||` and `==` with patterns. These file tests work in both.
- `[ ... ]` is a command: spaces around every bracket and operator are required, and quote variables.
- `-e` follows symlinks (a dangling link is NOT `-e` but IS `-L`).

In [5]:
%%bash
cd "$ROOT"
cd proj
f=src/a/main.py
[ -e "$f" ] && echo "exists"
[ -f "$f" ] && echo "regular file"
[ -d src ] && echo "src is a directory"
[ -f src ] || echo "src is not a regular file"
[ -s "$f" ] && echo "non-empty"
[ -s tests/a/test_main.py ] || echo "test_main.py is empty (-s false)"
[ -r "$f" ] && [ -w "$f" ] && echo "readable and writable"
[ -x "$f" ] || echo "not executable"
[ -e nothing ] || echo "missing path is not -e"
cd ..

exists
regular file
src is a directory
src is not a regular file
non-empty
test_main.py is empty (-s false)
readable and writable
not executable
missing path is not -e


## 04. `ls`: `-l`, `-la`, `-ld` (reading it safely)

- `ls -l` columns: permissions, link count, owner, group, size, date, name. Owner and date change per machine and per day.
- So in notes and scripts we pick out only stable columns, e.g. with `awk '{print $1, $NF}'` (permissions and name; `$NF` breaks on names with spaces, so this is only for a quick look).
- `ls -la` includes dotfiles; `-A` includes them but hides `.` and `..`; `-d` lists a directory itself, not its contents.
- `total N` is the first line of `ls -l` (disk blocks); `tail -n +2` drops it.
- Never parse `ls` for logic; use globs or `find`.

In [6]:
%%bash
cd "$ROOT"
cd proj
echo "--- permissions and names only:"
ls -l docs | tail -n +2 | awk '{print $1, $NF}'
echo "--- the directory itself (-ld), perms column only:"
ls -ld docs | awk '{print $1}'
echo "--- hidden files:"
touch .hidden
ls | tr '\n' ' '; echo
ls -A | tr '\n' ' '; echo
rm .hidden
cd ..

--- permissions and names only:
-rw-r--r-- README.md
drwxr-xr-x a
drwxr-xr-x b
--- the directory itself (-ld), perms column only:
drwxr-xr-x
--- hidden files:
docs empty_dir src tests 
.hidden docs empty_dir src tests 


## 05. `cp`: copying

- `cp src dst` copies a file. `-r` (or `-R`) copies directories recursively. `-p` keeps mode and times. `-n` never overwrite. `-i` asks first.
- If `dst` is an existing directory the file goes INSIDE it. If `dst` does not exist it becomes the new name.
- `cp -r dir newdir`: if `newdir` already exists the copy lands at `newdir/dir`. Run it twice and you get a surprise (shown below).
- GNU extras like `cp -a`, `--parents` are not all on BSD; `-R -p` is the portable pair (`cp -a` also exists on macOS).

In [7]:
%%bash
cd "$ROOT"
cd proj
cp src/a/main.py /dev/null          # copy to a safe sink, just to show it runs
cp src/a/main.py docs/copied.py
ls docs
echo "--- -n never overwrites (silently skips):"
echo "ORIGINAL" > docs/target.txt
echo "NEW"      > docs/new.txt
cp -n docs/new.txt docs/target.txt
cat docs/target.txt
echo "--- -p keeps modification time:"
touch -t 202001010000 docs/new.txt
cp docs/new.txt docs/plain.txt
cp -p docs/new.txt docs/kept.txt
ls -l docs/kept.txt | awk '{print "kept:", $6, $7, $8}'
echo "plain copy got a new year: $(stat -f '%Sm' -t %Y docs/plain.txt | awk '{print ($1 == 2020) ? "no, still 2020" : "yes, today\x27s year"}')"
cd ..

README.md
a
b
copied.py
--- -n never overwrites (silently skips):
ORIGINAL
--- -p keeps modification time:
kept: Jan 1 2020
plain copy got a new year: yes, today's year


In [8]:
%%bash
cd "$ROOT"
echo "--- directory copy twice: second time nests (bad vs good)"
mkdir -p cpdemo/orig && : > cpdemo/orig/f.txt
cp -r cpdemo/orig cpdemo/copy
cp -r cpdemo/orig cpdemo/copy        # copy now exists -> copies into it
find cpdemo | sort
rm -r cpdemo

--- directory copy twice: second time nests (bad vs good)
cpdemo
cpdemo/copy
cpdemo/copy/f.txt
cpdemo/copy/orig
cpdemo/copy/orig/f.txt
cpdemo/orig
cpdemo/orig/f.txt


## 06. `mv`: moving and renaming

- `mv a b` renames or moves; on the same disk it is instant and keeps the inode (and permissions).
- `mv file dir/` moves INTO a directory (the slash makes your intent clear and fails if `dir` is missing).
- `-n` do not overwrite, `-i` ask, `-f` force. Without these, `mv` silently replaces an existing target.
- Many files: `mv a b c destdir/` (last argument is the destination). Use `--` before names that start with `-`.

In [9]:
%%bash
cd "$ROOT"
mkdir mvdemo && cd mvdemo
echo A > a.txt; echo B > b.txt
mv a.txt renamed.txt
ls
echo "--- silent overwrite (no -n):"
mv b.txt renamed.txt; cat renamed.txt; ls
echo "--- destination slash on a missing dir fails safely:"
echo C > c.txt
mv c.txt nodir/ 2>&1 | sed 's/^mv: //' || true
ls
echo "--- move several into a directory:"
mkdir box; echo 1 > one; echo 2 > two
mv one two box/
ls box
cd .. && rm -r mvdemo

b.txt
renamed.txt
--- silent overwrite (no -n):
B
renamed.txt
--- destination slash on a missing dir fails safely:
rename c.txt to nodir/: No such file or directory
c.txt
renamed.txt
--- move several into a directory:
one
two


## 07. `rm`: removing, and the safety habits

- `rm f` removes a file (no trash, no undo). `-r` recurse, `-f` no questions and no error if missing, `-i` ask for each, `-d` empty directory (like `rmdir`), `rmdir` only empty directories.
- Habit 1: preview with `echo rm ...` or `ls` first; glob output is what `rm` will get.
- Habit 2: guard variables: `rm -rf "${dir:?}"/*` aborts if `dir` is empty or unset.
- Habit 3: use `--` before a list so names starting with `-` are not read as options.
- Habit 4: avoid `rm -rf $var` unquoted. A space in the value splits it into two deletes.

In [10]:
%%bash
cd "$ROOT"
mkdir rmdemo && cd rmdemo
touch a.tmp b.tmp keep.txt
echo "--- preview what the glob matches:"
echo rm *.tmp
rm -- *.tmp
ls
echo "--- rm -f on a missing file is quiet and succeeds:"
rm -f nothing.txt; echo "status $?"
echo "--- rm on a missing file without -f:"
rm nothing.txt 2>&1 | sed 's/^rm: //'
echo "--- file starting with a dash:"
touch ./-weird
rm -- -weird
echo "--- rmdir only removes empty directories:"
mkdir full empty && touch full/x
rmdir empty && echo "empty removed"
rmdir full 2>&1 | sed 's/^rmdir: //'
rm -r full
cd .. && rm -r rmdemo

--- preview what the glob matches:
rm a.tmp b.tmp
keep.txt
--- rm -f on a missing file is quiet and succeeds:
status 0
--- rm on a missing file without -f:
nothing.txt: No such file or directory
--- file starting with a dash:
--- rmdir only removes empty directories:
empty removed
full: Directory not empty


In [11]:
%%bash
cd "$ROOT"
echo "--- the guard: an empty variable aborts instead of becoming rm -rf /*"
( dir=""; rm -rf "${dir:?refusing: dir is empty}"/* ) 2>&1 | sed 's/^[^:]*: line [0-9]*: //'
echo "(the subshell stopped before rm ran)"
echo "--- unquoted variable splits on spaces (shown with echo, NOT run):"
dir="my files"
echo "unquoted -> rm gets these arguments:"; printf '  [%s]\n' $dir
echo "quoted   -> rm gets this argument:";  printf '  [%s]\n' "$dir"

--- the guard: an empty variable aborts instead of becoming rm -rf /*
dir: refusing: dir is empty
(the subshell stopped before rm ran)
--- unquoted variable splits on spaces (shown with echo, NOT run):
unquoted -> rm gets these arguments:
  [my]
  [files]
quoted   -> rm gets this argument:
  [my files]


## 08. Globbing in depth

- The shell expands patterns BEFORE the command runs: `*` any string, `?` one char, `[abc]` / `[a-c]` / `[!abc]` a set.
- A glob that matches nothing stays as the literal text (bash default), which can confuse commands. `shopt -s nullglob` makes it expand to nothing.
- `*` does NOT match names starting with `.`; `.[!.]*` matches dotfiles without `.` and `..` (it misses names that start with two dots).
- Globs never cross `/`. Case matters unless `shopt -s nocaseglob`. Ranges like `[a-c]` depend on the locale (bytewise here, `LC_ALL=C`). Note that the macOS default disk is case-insensitive, so `a.txt` and `A.TXT` cannot both exist there.
- Quote when you do NOT want expansion: `grep 'a*b'`.

In [12]:
%%bash
cd "$ROOT"
mkdir g && cd g
touch a.txt b.txt c.md Notes.TXT .hidden file1 file2 file10 x
echo "*       :" *
echo "*.txt   :" *.txt
echo "?       (single-char names):" ?
echo "file?   :" file?
echo "file??  :" file??
echo "[ab].txt:" [ab].txt
echo "[!ab].*   :" [!ab].*
echo "[a-c]*  :" [a-c]*
echo "dotfiles:" .[!.]*
echo "no match stays literal:" *.zzz
echo "quoted:" "*.txt"

*       : Notes.TXT a.txt b.txt c.md file1 file10 file2 x
*.txt   : a.txt b.txt
?       (single-char names): x
file?   : file1 file2
file??  : file10
[ab].txt: a.txt b.txt
[!ab].*   : c.md
[a-c]*  : a.txt b.txt c.md
dotfiles: .hidden
no match stays literal: *.zzz
quoted: *.txt


## 09. `shopt` and extended globs on bash 3.2

- `shopt -s nullglob`: unmatched globs vanish. `dotglob`: `*` also matches dotfiles. `nocaseglob`: ignore case. `extglob`: `!(pat)`, `@(a|b)`, `+(pat)`, `?(pat)`, `*(pat)`.
- `globstar` (`**` for recursion) needs bash 4+: here `shopt globstar` is not a valid option.
- `extglob` patterns must be switched on in an EARLIER command than the line that uses them, because bash parses a line before running it.
- Turn options off again so later cells are not surprised.

In [13]:
%%bash
cd "$ROOT"
cd g
shopt -s nullglob
echo "nullglob, no match -> [$(echo *.zzz)]"
shopt -u nullglob
shopt -s dotglob
echo "dotglob: " *
shopt -u dotglob
shopt -s nocaseglob
echo "nocaseglob *.txt: " *.txt
shopt -u nocaseglob
shopt -s extglob
echo "extglob !(*.txt):  " !(*.txt)
echo "extglob @(a|b).txt:" @(a|b).txt
echo "extglob file+([0-9]):" file+([0-9])
shopt -u extglob
echo "--- globstar on 3.2:"
shopt globstar 2>&1 | sed 's/^[^:]*: line [0-9]*: //;s/^[^:]*: //'
cd ..

nullglob, no match -> []
dotglob:  .hidden Notes.TXT a.txt b.txt c.md file1 file10 file2 x
nocaseglob *.txt:  Notes.TXT a.txt b.txt
extglob !(*.txt):   Notes.TXT c.md file1 file10 file2 x
extglob @(a|b).txt: a.txt b.txt
extglob file+([0-9]): file1 file10 file2
--- globstar on 3.2:
globstar: invalid shell option name


## 10. `find`: the basics

- `find START TESTS ACTIONS`. Default action is `-print`. Output order is unspecified: pipe to `sort`.
- Tests: `-name 'glob'` (quote it!), `-iname` case-insensitive, `-type f|d|l`, `-size +1k`, `-mtime +7` (older than 7 days), `-newer file`, `-perm`, `-empty`.
- Depth: `-maxdepth N`, `-mindepth N` (both work on BSD find; GNU find warns if they come after other tests).
- Combine with `-a` (and, default), `-o` (or), `!` (not), and `\( \)` for grouping (escape the parentheses).

In [14]:
%%bash
cd "$ROOT"
cd proj
echo "--- *.py files:"; find . -type f -name '*.py' | sort
echo "--- directories one level down only:"; find . -maxdepth 1 -mindepth 1 -type d | sort
echo "--- python OR markdown:"; find . -type f \( -name '*.py' -o -name '*.md' \) | sort
echo "--- not python:"; find . -type f ! -name '*.py' | sort
echo "--- empty files and empty directories:"; find . -empty | sort
echo "--- case-insensitive:"; find . -iname 'readme*' | sort
cd ..

--- *.py files:
./docs/copied.py
./src/a/main.py
./src/b/util.py
./tests/a/test_main.py
--- directories one level down only:
./docs
./empty_dir
./src
./tests
--- python OR markdown:
./docs/README.md
./docs/copied.py
./src/a/main.py
./src/b/util.py
./tests/a/test_main.py
--- not python:
./docs/README.md
./docs/a/notes.txt
./docs/kept.txt
./docs/new.txt
./docs/plain.txt
./docs/target.txt
--- empty files and empty directories:
./docs/b
./empty_dir
./tests/a/test_main.py
./tests/b
--- case-insensitive:
./docs/README.md


## 11. `find -exec`: `\;` vs `+` vs `-print0 | xargs -0`

- `-exec cmd {} \;` runs `cmd` once PER file. Safe with spaces, slow for many files.
- `-exec cmd {} +` packs many files into a few calls. Same safety, much faster. `{}` must be last before `+`.
- `find ... -print0 | xargs -0 cmd` is the classic alternative: NUL-separated, safe with spaces and newlines.
- Never `for f in $(find ...)`: breaks on spaces. Never `find | xargs` without `-0` for odd names.
- The demo counts how many times the command ran.

In [15]:
%%bash
cd "$ROOT"
mkdir fx && cd fx
touch "a b.txt" c.txt d.txt
echo "--- \\; : one command per file"
find . -type f -exec echo "run:" {} \; | sort
echo "--- + : one command with all files (sh -c shows how many args one run got)"
find . -type f -exec sh -c 'echo "one run, $# files"' _ {} +
echo "--- -print0 | xargs -0 (spaces survive):"
find . -type f -print0 | xargs -0 -n1 echo "got:" | sort
echo "--- the broken way (unquoted, split on spaces):"
for f in $(find . -type f | sort); do echo "item: $f"; done
cd ..

--- \; : one command per file
run: ./a b.txt
run: ./c.txt
run: ./d.txt
--- + : one command with all files (sh -c shows how many args one run got)
one run, 3 files
--- -print0 | xargs -0 (spaces survive):
got: ./a b.txt
got: ./c.txt
got: ./d.txt
--- the broken way (unquoted, split on spaces):
item: ./a
item: b.txt
item: ./c.txt
item: ./d.txt


## 12. `find`: `-prune`, `-delete`, `-newer`, `-size`, `-perm`

- `-prune` skips a directory: `find . -path ./skip -prune -o -type f -print`.
- `-delete` removes matches. It works depth-first and implies `-depth`. Put it LAST; placed first it matches everything.
- `-newer file` selects files modified after `file`. `-size +1000c` means more than 1000 bytes (with `k` the size is rounded up to whole units, so prefer `c` for exact tests).
- `-perm 644` exactly, `-perm -u+x` at least these bits, `-mtime +30` older than 30 days (rounded in whole days).

In [16]:
%%bash
cd "$ROOT"
cd fx
mkdir -p skip keepme
touch skip/s1 keepme/k1
echo "--- prune the 'skip' directory:"
find . -path ./skip -prune -o -type f -print | sort
touch -t 202001010000 old.dat
touch -t 202101010000 mid.dat
touch -t 202201010000 new.dat
echo "--- .dat files newer than mid.dat:"; find . -type f -name '*.dat' -newer mid.dat | sort
printf 'x%.0s' $(seq 1 2000) > big.bin
echo "--- larger than 1000 bytes:"; find . -type f -size +1000c | sort
chmod 600 old.dat; chmod 755 new.dat
echo "--- perm exactly 600:"; find . -type f -perm 600 | sort
echo "--- executable by owner:"; find . -type f -perm -u+x | sort
echo "--- older than 30 days:"; find . -type f -mtime +30 -name '*.dat' | sort
cd ..

--- prune the 'skip' directory:
./a b.txt
./c.txt
./d.txt
./keepme/k1
--- .dat files newer than mid.dat:
./new.dat
--- larger than 1000 bytes:
./big.bin
--- perm exactly 600:
./old.dat
--- executable by owner:
./new.dat
--- older than 30 days:
./mid.dat
./new.dat
./old.dat


In [17]:
%%bash
cd "$ROOT"
echo "--- -delete placed LAST (safe) vs FIRST (matches everything):"
mkdir dd && cd dd && touch x.log y.txt
find . -type f -name '*.log' -delete
ls
touch x.log
find . -type f -delete -name '*.log'      # -delete runs before -name is tested
echo "after wrong order, left: [$(ls)]"
cd .. && rmdir dd

--- -delete placed LAST (safe) vs FIRST (matches everything):
y.txt
after wrong order, left: []


## 13. `chmod`, `umask` and `stat`

- `chmod 755 f`: owner rwx (7), group r-x (5), other r-x (5). Read 4, write 2, execute 1.
- Symbolic: `u+x`, `go-w`, `a=r`, `u=rw,g=r,o=`. Symbolic changes are relative; numeric sets exactly.
- For directories, `x` means "can enter/traverse". Without it you cannot `cd` or reach files inside.
- `umask 022` removes write for group/other from new items: files start at 666 -> 644, directories 777 -> 755.
- Read permissions: BSD `stat -f '%Lp %N' f` (GNU: `stat -c '%a %n' f`). `%Sp` gives text like `-rwxr-xr--`.
- `umask` set inside `( )` is local to that subshell.

In [18]:
%%bash
cd "$ROOT"
mkdir perm && cd perm
touch f
chmod 644 f;           stat -f '%Lp %Sp' f
chmod 755 f;           stat -f '%Lp %Sp' f
chmod u=rw,g=r,o= f;   stat -f '%Lp %Sp' f
chmod g+w f;           stat -f '%Lp' f
chmod go-rwx f;        stat -f '%Lp' f
chmod a+x f;           stat -f '%Lp' f
echo "--- umask effect on NEW files and directories:"
( umask 022; touch f022; mkdir d022; stat -f '%Lp %N' f022 d022 )
( umask 077; touch f077; mkdir d077; stat -f '%Lp %N' f077 d077 )
echo "--- directory without x cannot be entered:"
mkdir locked && touch locked/inside && chmod 600 locked
( cd locked ) 2>&1 | sed 's/^[^:]*: line [0-9]*: //;s/^[^:]*: //'
chmod 755 locked
cd ..

644 -rw-r--r--
755 -rwxr-xr-x
640 -rw-r-----
660
600
711
--- umask effect on NEW files and directories:
644 f022
755 d022
600 f077
700 d077
--- directory without x cannot be entered:
locked: Permission denied


## 14. Symlinks and hard links

- `ln -s target link` makes a symbolic link: a tiny file that stores a path. It can dangle, cross disks, and point to directories.
- `ln target link` makes a hard link: another NAME for the same data. Both names are equal; the data lives until the last name is removed. Cannot cross disks, and not allowed for directories.
- `[ -L link ]` tests for a symlink; `-e` follows it. `readlink link` prints where it points (one level).
- `readlink -f` and `realpath` resolve everything here, but are absent on older macOS: test with `command -v`. The portable `cd ... && pwd -P` resolves only the directory part, not a symlink file name.
- Link count is shown by `stat -f '%l'` (GNU `stat -c %h`).

In [19]:
%%bash
cd "$ROOT"
mkdir ln && cd ln
echo "data" > real.txt
ln -s real.txt soft.txt
ln real.txt hard.txt
echo "soft -> $(readlink soft.txt)"
echo "link count of real.txt: $(stat -f '%l' real.txt)"
echo "-L soft:"; [ -L soft.txt ] && echo yes
echo "-L hard:"; [ -L hard.txt ] || echo "no (a hard link is a normal file)"
echo "same file (-ef) hard:"; [ real.txt -ef hard.txt ] && echo yes
echo "--- delete the original:"
rm real.txt
cat hard.txt
cat soft.txt 2>&1 | sed 's/^cat: //'
echo "soft is -L: $( [ -L soft.txt ] && echo yes ), still -e: $( [ -e soft.txt ] && echo yes || echo no ) (dangling)"
echo "--- file type text:"
stat -f '%HT' soft.txt hard.txt

soft -> real.txt
link count of real.txt: 2
-L soft:
yes
-L hard:
no (a hard link is a normal file)
same file (-ef) hard:
yes
--- delete the original:
data
soft.txt: No such file or directory
soft is -L: yes, still -e: no (dangling)
--- file type text:
Symbolic Link
Regular File


In [20]:
%%bash
cd "$ROOT"
cd ln
echo "--- resolving the full path:"
echo "readlink -f:  $(command -v readlink >/dev/null && readlink -f hard.txt | sed 's|.*/||')"
echo "realpath:     $(command -v realpath >/dev/null && realpath hard.txt | sed 's|.*/||' || echo 'not installed')"
echo "portable way: $(cd "$(dirname hard.txt)" && pwd -P | sed 's|.*/||')/$(basename hard.txt)"
cd ..

--- resolving the full path:
readlink -f:  hard.txt
realpath:     hard.txt
portable way: ln/hard.txt


## 15. `touch` with fixed times, and time tests

- `touch f` creates the file or updates its times to now. `touch -t YYYYMMDDhhmm[.ss] f` sets a fixed time (great for repeatable tests).
- `touch -r ref f` copies the time of `ref`. GNU `touch -d '2 days ago'` fails on BSD/macOS (its `-d` only takes `YYYY-MM-DDThh:mm:SS`).
- `-nt` / `-ot` compare modification times; `find -newer` and `-mtime` use them too.
- Print a time deterministically with `stat -f '%Sm' -t '%Y-%m-%d %H:%M:%S' f` (GNU: `stat -c %y`). The sandbox uses `TZ=UTC`.

In [21]:
%%bash
cd "$ROOT"
mkdir tm && cd tm
touch -t 202001010000 old; touch -t 202101011230.45 newer
stat -f '%Sm' -t '%Y-%m-%d %H:%M:%S' old
stat -f '%Sm' -t '%Y-%m-%d %H:%M:%S' newer
[ newer -nt old ] && echo "newer is newer than old"
[ old -ot newer ] && echo "old is older than newer"
touch -r old copy_of_time
stat -f '%Sm' -t '%Y-%m-%d' copy_of_time
echo "--- GNU-style -d '2020-01-01' fails here:"
touch -d '2020-01-01' bad 2>/dev/null || echo "touch -d failed here (status $?)"
cd ..

2020-01-01 00:00:00
2021-01-01 12:30:45
newer is newer than old
old is older than newer
2020-01-01
--- GNU-style -d '2020-01-01' fails here:
touch -d failed here (status 1)


## 16. `basename`, `dirname` and path pieces

- `basename /a/b/c.txt` gives `c.txt`; `basename /a/b/c.txt .txt` strips the suffix too.
- `dirname /a/b/c.txt` gives `/a/b`. `dirname file` (no slash) gives `.`.
- Trailing slashes are ignored. These are commands (a process each); in loops, parameter expansion `${p##*/}` and `${p%/*}` is faster.
- Careful: `${p%/*}` on a name with no slash returns the name unchanged, unlike `dirname`.

In [22]:
%%bash
cd "$ROOT"
p=/a/b/c.txt
echo "basename: $(basename "$p")"
echo "no suffix: $(basename "$p" .txt)"
echo "dirname:  $(dirname "$p")"
echo "dirname of bare name: $(dirname c.txt)"
echo "trailing slash: $(basename /a/b/)"
echo "builtin forms: ${p##*/}  ${p%/*}  ${p%.txt}"
bare=c.txt
echo "bare with %/*: ${bare%/*}   <- unchanged, not '.'"

basename: c.txt
no suffix: c
dirname:  /a/b
dirname of bare name: .
trailing slash: b
builtin forms: c.txt  /a/b  /a/b/c
bare with %/*: c.txt   <- unchanged, not '.'


## 17. `du` and `df` (without machine-specific numbers)

- `du -sk dir` total size of `dir` in 1K blocks (portable). `-h` gives human units (K/M/G) on both BSD and GNU. `-s` summary, `-d N` (BSD) / `--max-depth=N` (GNU) limit depth.
- `df -k .` shows capacity of the filesystem that holds `.`; `df -h` human units.
- Sizes depend on block size and machine, so here we print only facts that are always true (non-zero, ordering).
- `wc -c < file` gives exact bytes; `stat -f %z file` too (GNU `stat -c %s`).

In [23]:
%%bash
cd "$ROOT"
cd fx
echo "exact byte size via stat:  $(stat -f '%z' big.bin)"
echo "exact byte size via wc:    $(wc -c < big.bin | tr -d ' ')"
du -sk . | awk '{print ($1 > 0) ? "du -sk: positive number of 1K blocks" : "zero?"}'
df -k . | awk 'NR==2 {print ($2 > 0) ? "df -k: filesystem has capacity" : "?"}'
cd ..

exact byte size via stat:  2000
exact byte size via wc:    2000
du -sk: positive number of 1K blocks
df -k: filesystem has capacity


## 18. `file(1)` and reading metadata

- `file name` guesses the type from content (not the extension): text, script, directory, empty, image...
- `-b` brief (no file name); `--mime-type` gives a type like `text/x-shellscript`.
- Use `stat` for exact numbers: size (`%z`), type (`%HT`), permissions (`%Lp`), link count (`%l`), mod time (`%Sm`).
- `file` text wording differs between versions; test with `grep -q` rather than comparing whole strings.

In [24]:
%%bash
cd "$ROOT"
mkdir fl && cd fl
printf '#!/bin/bash\necho hi\n' > script.sh
printf 'plain words here\n' > note.txt
: > empty
mkdir dir
file -b note.txt
file -b empty
file -b dir
file --mime-type -b script.sh
if file -b script.sh | grep -qi 'script'; then echo "script.sh looks like a script"; fi
echo "--- stat as a small report:"
stat -f '%N: %z bytes, type %HT, mode %Lp' note.txt
stat -f '%N: type %HT, mode %Lp' dir
cd ..

ASCII text
empty
directory
text/x-shellscript
script.sh looks like a script
--- stat as a small report:
note.txt: 17 bytes, type Regular File, mode 644
dir: type Directory, mode 755


## 19. `mktemp -d` and `trap` cleanup (fresh shell)

- `mktemp -d` makes a unique empty directory and prints its path. Never invent `/tmp/myscript.$$` names by hand.
- `trap 'rm -rf "$tmp"' EXIT` runs the cleanup when the script exits for almost any reason (normal, `exit`, error under `set -e`, Ctrl-C). It cannot run after `kill -9`.
- Put the trap right after `mktemp` so there is no window without cleanup.
- macOS `mktemp -d` works without a template; GNU too. A portable template form is `mktemp -d "${TMPDIR:-/tmp}/name.XXXXXX"`.
- Shown in a fresh shell because `trap ... EXIT` would otherwise fire when the notebook session ends.

In [25]:
%%bash fresh
tmp=$(mktemp -d "${TMPDIR:-/tmp}/demo.XXXXXX") || exit 1
trap 'rm -rf "$tmp"; echo "cleanup ran"' EXIT
echo "created: $( [ -d "$tmp" ] && echo yes)"
echo hello > "$tmp/work.txt"
cat "$tmp/work.txt"
exit 0

created: yes
hello
cleanup ran


In [26]:
%%bash
cd "$ROOT"
echo "--- the same cleanup runs on an early exit too (a separate bash -c process):"
bash -c 'tmp=$(mktemp -d); trap "rm -rf \"$tmp\"; echo cleanup-on-failure" EXIT; touch "$tmp/x"; exit 3'
echo "inner status: $?"

--- the same cleanup runs on an early exit too (a separate bash -c process):
cleanup-on-failure
inner status: 3


## 20. Disk-safety checklist

- Quote every path variable: `"$dir"`.
- Use `${var:?message}` before destructive commands.
- `--` ends option parsing, so odd file names cannot become flags.
- Preview with `echo`, `ls` or `find ... -print` before `rm` or `-delete`; then replace the preview with the real command.
- Work in a variable-defined sandbox (`mktemp -d`) and never `cd` somewhere unchecked: `cd "$dir" || exit 1`.
- Prefer `find ... -exec ... +` or `-print0 | xargs -0` over parsing `ls`.
- `cd` failing inside `cd x && rm -rf *` is safe because of `&&`; `cd x; rm -rf *` is not.

In [27]:
%%bash
cd "$ROOT"
mkdir safe && cd safe
touch a b
target=nope
echo "--- bad: ; keeps going after a failed cd (we only PRINT what rm would do):"
cd "$target" 2>/dev/null; echo "would run: rm -rf * in $(basename "$(pwd)")"
echo "--- good: && stops:"
cd "$target" 2>/dev/null && echo "would run rm" || echo "cd failed, nothing runs"
echo "--- good: explicit guard:"
cd "$target" 2>/dev/null || echo "guard: cd failed, skip cleanup"
cd .. && rm -r safe

--- bad: ; keeps going after a failed cd (we only PRINT what rm would do):
would run: rm -rf * in safe
--- good: && stops:
cd failed, nothing runs
--- good: explicit guard:
guard: cd failed, skip cleanup


## 21. Cleanup

- Remove the whole sandbox tree (`$ROOT`) with the guarded form from section 07.

In [28]:
%%bash
cd "$ROOT"
cd "$ROOT/.." && rm -rf "${ROOT:?}"
echo cleaned
unset ROOT

cleaned
